# ESTIMACION PROBABILISTICA	 SEMANAL DE EVENTOS SISMICOS POR MEDIO DE REDES NEURONALES A PARTIR DE DATA HISTORICA E INSTRUMENTAL DEL INSTITUTO GEOFISICO DEL PERU
## Objetivos
Determinar la eficacia de un modelo de red neuronal artificial para el pronostico probabilistico semanal de eventos sismicos de magnitud igual o superior a 4.0 para el año 2026 en Peru.

## Carga de datos

In [ ]:
from pathlib import Path
import pandas as pd

# Archivos fuente ubicados en la carpeta datos IGP
carpeta_datos = Path("datos IGP")
archivo_historico = carpeta_datos / "datos-sismicos_Historico_1471-01-01_1959-12-31.xlsx"
archivo_instrumental = carpeta_datos / "datos-sismicos_Instrumental_1960-01-01_2026-10-01.xlsx"

# Lectura de los catÃ¡logos (primera hoja de cada Excel)
datos_historicos = pd.read_excel(archivo_historico)
datos_instrumentales = pd.read_excel(archivo_instrumental)

print(f"CatÃ¡logo histÃ³rico: {datos_historicos.shape[0]:,} filas x {datos_historicos.shape[1]} columnas")
display(datos_historicos.head())
print(f"CatÃ¡logo instrumental: {datos_instrumentales.shape[0]:,} filas x {datos_instrumentales.shape[1]} columnas")
display(datos_instrumentales.head())

In [ ]:
from pathlib import Path
 import pandas as pd
 
 carpeta_datos = Path("datos IGP")
 datos_sismicos = {
     "historico": pd.read_excel(carpeta_datos / "datos-sismicos_Historico_1471-01-01_1959-12-31.xlsx"),
     "instrumental": pd.read_excel(carpeta_datos / "datos-sismicos_Instrumental_1960-01-01_2026-10-01.xlsx")
 }
 for nombre, df in datos_sismicos.items():
     print(f"{nombre}: {df.shape[0]:,} filas x {df.shape[1]} columnas")
     display(df.head())


## EDA y limpieza
 El análisis revisa estructura, tipos, valores faltantes, marcadores `-`, duplicados y fechas. La limpieza conserva copias originales y crea versiones depuradas para análisis.

In [ ]:
import pandas as pd
from IPython.display import display

# Perfil inicial: dimensiones, tipos y valores faltantes por variable
for nombre, df in datos_sismicos.items():
    print(f"\n{nombre.upper()} — {df.shape[0]:,} filas, {df.shape[1]} columnas")
    display(pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "faltantes": df.isna().sum(),
        "% faltantes": (100 * df.isna().mean()).round(1)
    }))
    print("Filas duplicadas exactas:", int(df.duplicated().sum()))
    display(df.head(3))

# Limpieza conservadora: normalizar textos y marcadores de ausencia, convertir magnitudes/coordenadas/profundidad,
# detectar fechas inválidas y retirar únicamente filas duplicadas exactas.
datos_limpios = {}
resumen_limpieza = []
for nombre, original in datos_sismicos.items():
    df = original.copy()
    for col in df.select_dtypes(include="object"):
        df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)
        df[col] = df[col].replace({"": pd.NA, "-": pd.NA, "—": pd.NA, "NA": pd.NA, "N/A": pd.NA})
    numericas = [c for c in df.columns if any(t in str(c).lower() for t in ("latitud", "longitud", "profundidad", "magnitud"))]
    for col in numericas:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    columna_fecha = next((c for c in df.columns if "fecha" in str(c).lower()), None)
    fechas_invalidas = 0
    if columna_fecha:
        fechas = pd.to_datetime(df[columna_fecha], errors="coerce")
        fechas_invalidas = int(fechas.isna().sum())
        df[columna_fecha] = fechas
    duplicadas = int(df.duplicated().sum())
    df = df.drop_duplicates().reset_index(drop=True)
    datos_limpios[nombre] = df
    resumen_limpieza.append({
        "catalogo": nombre, "filas originales": len(original), "filas limpias": len(df),
        "duplicados retirados": duplicadas, "fechas inválidas": fechas_invalidas,
        "faltantes tras normalizar": int(df.isna().sum().sum())
    })
display(pd.DataFrame(resumen_limpieza).set_index("catalogo"))

# Tres insights calculados directamente a partir de ambos catálogos
h, i = datos_sismicos["historico"], datos_sismicos["instrumental"]
print("INSIGHT 1 — Volumen: el catálogo histórico tiene {:,} registros y el instrumental {:,} ({} veces más).".format(len(h), len(i), round(len(i) / max(len(h), 1), 1)))

coberturas = []
for nombre, df in (("histórico", h), ("instrumental", i)):
    columna_fecha = next((c for c in df.columns if "fecha" in str(c).lower()), None)
    if columna_fecha:
        fechas = pd.to_datetime(df[columna_fecha], errors="coerce").dropna()
        if not fechas.empty:
            coberturas.append(f"{nombre}: {fechas.min().date()}–{fechas.max().date()}")
print("INSIGHT 2 — Cobertura temporal: " + "; ".join(coberturas) + ".")

faltantes_magnitud = []
for nombre, df in (("histórico", h), ("instrumental", i)):
    columnas_magnitud = [c for c in df.columns if "magnitud" in str(c).lower()]
    if columnas_magnitud:
        porcentaje = 100 * df[columnas_magnitud].isna().all(axis=1).mean()
        faltantes_magnitud.append(f"{nombre}: {porcentaje:.1f}% sin magnitud en todos sus campos")
print("INSIGHT 3 — Completitud de magnitud: " + "; ".join(faltantes_magnitud) + ".")
# Los DataFrames originales siguen en datos_sismicos; usar datos_limpios para los análisis siguientes.
for nombre, df in datos_limpios.items():
    print(f"{nombre} limpio: {df.shape}")

